In [1]:
import pyspark
from pyspark.sql.functions import * 
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('MapReduce').getOrCreate()

22/02/26 08:57:45 WARN Utils: Your hostname, vania-Latitude-7400 resolves to a loopback address: 127.0.1.1; using 192.168.0.16 instead (on interface wlo1)
22/02/26 08:57:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
22/02/26 08:57:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
import numpy as np
import matplotlib.pyplot as plt
import nltk
import string
from sentiment_analysis_spanish import sentiment_analysis

#### Lectura

In [3]:
rdd_letras = spark.sparkContext.textFile('../data/top_10_mexico.csv').map(lambda x: x.split('*'))

#### Limpieza

In [4]:
rdd_letras = rdd_letras.map(lambda x: (x[0], x[1], x[2].lower()))
rdd_letras = rdd_letras.map(lambda x: (x[0], x[1], x[2].split(' ')))
signos_puntuacion = list(string.punctuation) + ['¡', '¿']

def elimina_puntuacion(letra):
    letra_limpia = [''.join(caracter for caracter in palabra if caracter not in signos_puntuacion) for palabra in letra] 
    return letra_limpia

rdd_letras = rdd_letras.map(lambda x: (x[0], x[1], elimina_puntuacion(x[2])))

In [5]:
palabras_vacias = nltk.corpus.stopwords.words('spanish') + ['']

def quita_palabras_vacias(letra):
    letra_limpia = [palabra for palabra in letra if palabra not in palabras_vacias]
    return letra_limpia

rdd_letras = rdd_letras.map(lambda x: (x[0], x[1], quita_palabras_vacias(x[2])))

#### flatMapValues

In [6]:
x = spark.sparkContext.parallelize([("a", ["x", "y", "z"]), ("b", ["p", "r"])])

In [7]:
x.flatMapValues(lambda x: x).collect()

[('a', 'x'), ('a', 'y'), ('a', 'z'), ('b', 'p'), ('b', 'r')]

#### Para el ejemplo

In [26]:
rdd_letras.collect()

[('Bad Bunny, Jhay Cortez',
  'DÁKITI',
  ['baby',
   'enteré',
   'nota',
   've',
   'ahí',
   'llegao',
   'sabes',
   'llevaré',
   'dime',
   'quieres',
   'beber',
   'bebé',
   'quién',
   'va',
   'hablar',
   'si',
   'dejamos',
   'ver',
   'veces',
   'dolce',
   'veces',
   'bulgari',
   'quito',
   'después',
   'parties',
   'copas',
   'vino',
   'libras',
   'mari',
   'bien',
   'suelta',
   'safari',
   'mueve',
   'culo',
   'fenomenal',
   'pa',
   'devorarte',
   'animal',
   'si',
   'venío',
   'vo',
   'esperar',
   'cama',
   'vo',
   'celebrar',
   'baby',
   'opongo',
   'siempre',
   'pongo',
   'si',
   'tiras',
   'vamo',
   'nadar',
   'hondo',
   'si',
   'pongo',
   'septiembre',
   'agosto',
   'cojone',
   'digan',
   'amigas',
   'enteré',
   'nota',
   've',
   'ahí',
   'llegao',
   'sabes',
   'llevaré',
   'dime',
   'quieres',
   'beber',
   'bebé',
   'quién',
   'va',
   'hablar',
   'si',
   'dejamos',
   'ver',
   'sigues',
   'mami',
   'ju

In [25]:
df = rdd_letras.toDF() 
df.show()

+--------------------+--------------------+--------------------+
|                  _1|                  _2|                  _3|
+--------------------+--------------------+--------------------+
|Bad Bunny, Jhay C...|              DÁKITI|[baby, enteré, no...|
|             BICHOTA|             Karol G|[salgo, acicala, ...|
|             Bandido|   Myke Towers, Juhn|[buena, gustan, m...|
|  LA NOCHE DE ANOCHE|  Bad Bunny, Rosalía|[sé, volverá, pas...|
|               Hawái|              Maluma|[deja, mentirte, ...|
|       Baila conmigo|Selena Gómez, Rau...|[bebé, sé, si, ha...|
|               Reloj|Anuel AA, Rauw Al...|[real, muerte, oí...|
|           Ropa Cara|              Camilo|[historia, basada...|
|        Hecha Pa' Mi|                Boza|[mirada, encanta,...|
|         Algo mágico|      Rauw Alejandro|[rarauw, cama, ah...|
+--------------------+--------------------+--------------------+



In [9]:
df = rdd_letras.toDF() 
df = df.select('_1', '_3')

In [10]:
df.show(2)

+--------------------+--------------------+
|                  _1|                  _3|
+--------------------+--------------------+
|Bad Bunny, Jhay C...|[baby, enteré, no...|
|             BICHOTA|[salgo, acicala, ...|
+--------------------+--------------------+
only showing top 2 rows



In [11]:
rdd_aux = df.rdd

In [13]:
rdd_aux.collect()

[Row(_1='Bad Bunny, Jhay Cortez', _3=['baby', 'enteré', 'nota', 've', 'ahí', 'llegao', 'sabes', 'llevaré', 'dime', 'quieres', 'beber', 'bebé', 'quién', 'va', 'hablar', 'si', 'dejamos', 'ver', 'veces', 'dolce', 'veces', 'bulgari', 'quito', 'después', 'parties', 'copas', 'vino', 'libras', 'mari', 'bien', 'suelta', 'safari', 'mueve', 'culo', 'fenomenal', 'pa', 'devorarte', 'animal', 'si', 'venío', 'vo', 'esperar', 'cama', 'vo', 'celebrar', 'baby', 'opongo', 'siempre', 'pongo', 'si', 'tiras', 'vamo', 'nadar', 'hondo', 'si', 'pongo', 'septiembre', 'agosto', 'cojone', 'digan', 'amigas', 'enteré', 'nota', 've', 'ahí', 'llegao', 'sabes', 'llevaré', 'dime', 'quieres', 'beber', 'bebé', 'quién', 'va', 'hablar', 'si', 'dejamos', 'ver', 'sigues', 'mami', 'juqueao', 'si', 'uru', 'parqueao', 'dando', 'vueltas', 'condado', 'contigo', 'siempre', 'arrebatao', 'señora', 'toma', 'cinco', 'mil', 'gástalo', 'sephora', 'louis', 'vuitton', 'compra', 'pandora', 'piercing', 'hombres', 'perfora', 'eheheh', 'hace

In [15]:
rdd_palabras = rdd_aux.flatMapValues(lambda x: x)

In [16]:
rdd_palabras.collect()

[('Bad Bunny, Jhay Cortez', 'baby'),
 ('Bad Bunny, Jhay Cortez', 'enteré'),
 ('Bad Bunny, Jhay Cortez', 'nota'),
 ('Bad Bunny, Jhay Cortez', 've'),
 ('Bad Bunny, Jhay Cortez', 'ahí'),
 ('Bad Bunny, Jhay Cortez', 'llegao'),
 ('Bad Bunny, Jhay Cortez', 'sabes'),
 ('Bad Bunny, Jhay Cortez', 'llevaré'),
 ('Bad Bunny, Jhay Cortez', 'dime'),
 ('Bad Bunny, Jhay Cortez', 'quieres'),
 ('Bad Bunny, Jhay Cortez', 'beber'),
 ('Bad Bunny, Jhay Cortez', 'bebé'),
 ('Bad Bunny, Jhay Cortez', 'quién'),
 ('Bad Bunny, Jhay Cortez', 'va'),
 ('Bad Bunny, Jhay Cortez', 'hablar'),
 ('Bad Bunny, Jhay Cortez', 'si'),
 ('Bad Bunny, Jhay Cortez', 'dejamos'),
 ('Bad Bunny, Jhay Cortez', 'ver'),
 ('Bad Bunny, Jhay Cortez', 'veces'),
 ('Bad Bunny, Jhay Cortez', 'dolce'),
 ('Bad Bunny, Jhay Cortez', 'veces'),
 ('Bad Bunny, Jhay Cortez', 'bulgari'),
 ('Bad Bunny, Jhay Cortez', 'quito'),
 ('Bad Bunny, Jhay Cortez', 'después'),
 ('Bad Bunny, Jhay Cortez', 'parties'),
 ('Bad Bunny, Jhay Cortez', 'copas'),
 ('Bad Bunny, 

In [17]:
rddInverted = rdd_palabras.map(lambda x: (x[1], x[0]))

In [19]:
rddInverted.collect()

[('baby', 'Bad Bunny, Jhay Cortez'),
 ('enteré', 'Bad Bunny, Jhay Cortez'),
 ('nota', 'Bad Bunny, Jhay Cortez'),
 ('ve', 'Bad Bunny, Jhay Cortez'),
 ('ahí', 'Bad Bunny, Jhay Cortez'),
 ('llegao', 'Bad Bunny, Jhay Cortez'),
 ('sabes', 'Bad Bunny, Jhay Cortez'),
 ('llevaré', 'Bad Bunny, Jhay Cortez'),
 ('dime', 'Bad Bunny, Jhay Cortez'),
 ('quieres', 'Bad Bunny, Jhay Cortez'),
 ('beber', 'Bad Bunny, Jhay Cortez'),
 ('bebé', 'Bad Bunny, Jhay Cortez'),
 ('quién', 'Bad Bunny, Jhay Cortez'),
 ('va', 'Bad Bunny, Jhay Cortez'),
 ('hablar', 'Bad Bunny, Jhay Cortez'),
 ('si', 'Bad Bunny, Jhay Cortez'),
 ('dejamos', 'Bad Bunny, Jhay Cortez'),
 ('ver', 'Bad Bunny, Jhay Cortez'),
 ('veces', 'Bad Bunny, Jhay Cortez'),
 ('dolce', 'Bad Bunny, Jhay Cortez'),
 ('veces', 'Bad Bunny, Jhay Cortez'),
 ('bulgari', 'Bad Bunny, Jhay Cortez'),
 ('quito', 'Bad Bunny, Jhay Cortez'),
 ('después', 'Bad Bunny, Jhay Cortez'),
 ('parties', 'Bad Bunny, Jhay Cortez'),
 ('copas', 'Bad Bunny, Jhay Cortez'),
 ('vino', 'Bad

In [20]:
rdd_ = rddInverted.combineByKey(lambda x: [x], lambda x,y:x+[y], lambda x,y:x+y)
rdd_.collect()

[('ve',
  ['Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Hawái',
   'Ropa Cara']),
 ('llegao',
  ['Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez']),
 ('llevaré',
  ['Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez']),
 ('dime',
  ['Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Bandido',
   'LA NOCHE DE ANOCHE',
   'LA NOCHE DE ANOCHE']),
 ('quieres',
  ['Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Baila conmigo',
   'Baila conmigo',
   'Algo mágico']),
 ('quién',
  ['Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'BICHOTA',
   'Bandido',
   'LA NOCHE DE ANOCHE',
   'Hawái',
   'Hawái']),
 ('va',
  ['Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Bad Bunny, Jhay Cortez',
   'Hawái',
   'Hawái',
   'Hawái',
   'Hawái',
   'Hawái',
   'Hawái',
   'R

In [21]:
rdd_.map(lambda x: (x[0], list(dict.fromkeys(x[1])))).collect()

[('ve', ['Bad Bunny, Jhay Cortez', 'Hawái', 'Ropa Cara']),
 ('llegao', ['Bad Bunny, Jhay Cortez']),
 ('llevaré', ['Bad Bunny, Jhay Cortez']),
 ('dime', ['Bad Bunny, Jhay Cortez', 'Bandido', 'LA NOCHE DE ANOCHE']),
 ('quieres', ['Bad Bunny, Jhay Cortez', 'Baila conmigo', 'Algo mágico']),
 ('quién',
  ['Bad Bunny, Jhay Cortez',
   'BICHOTA',
   'Bandido',
   'LA NOCHE DE ANOCHE',
   'Hawái']),
 ('va', ['Bad Bunny, Jhay Cortez', 'Hawái', 'Reloj']),
 ('si',
  ['Bad Bunny, Jhay Cortez',
   'BICHOTA',
   'Bandido',
   'LA NOCHE DE ANOCHE',
   'Hawái',
   'Baila conmigo',
   'Reloj',
   "Hecha Pa' Mi",
   'Algo mágico']),
 ('dejamos', ['Bad Bunny, Jhay Cortez']),
 ('dolce', ['Bad Bunny, Jhay Cortez']),
 ('quito', ['Bad Bunny, Jhay Cortez']),
 ('mari', ['Bad Bunny, Jhay Cortez', 'BICHOTA']),
 ('bien', ['Bad Bunny, Jhay Cortez', 'Bandido', 'Hawái', 'Ropa Cara']),
 ('mueve', ['Bad Bunny, Jhay Cortez', 'Reloj']),
 ('fenomenal', ['Bad Bunny, Jhay Cortez']),
 ('pa',
  ['Bad Bunny, Jhay Cortez',
   

#### Solución completa

In [22]:
rdd_aux.flatMapValues(lambda x: x) \
    .map(lambda x: (x[1], x[0])) \
    .combineByKey(lambda x: [x], lambda x,y:x+[y], lambda x,y:x+y) \
    .map(lambda x: (x[0], list(dict.fromkeys(x[1])))).collect()

[('ve', ['Bad Bunny, Jhay Cortez', 'Hawái', 'Ropa Cara']),
 ('llegao', ['Bad Bunny, Jhay Cortez']),
 ('llevaré', ['Bad Bunny, Jhay Cortez']),
 ('dime', ['Bad Bunny, Jhay Cortez', 'Bandido', 'LA NOCHE DE ANOCHE']),
 ('quieres', ['Bad Bunny, Jhay Cortez', 'Baila conmigo', 'Algo mágico']),
 ('quién',
  ['Bad Bunny, Jhay Cortez',
   'BICHOTA',
   'Bandido',
   'LA NOCHE DE ANOCHE',
   'Hawái']),
 ('va', ['Bad Bunny, Jhay Cortez', 'Hawái', 'Reloj']),
 ('si',
  ['Bad Bunny, Jhay Cortez',
   'BICHOTA',
   'Bandido',
   'LA NOCHE DE ANOCHE',
   'Hawái',
   'Baila conmigo',
   'Reloj',
   "Hecha Pa' Mi",
   'Algo mágico']),
 ('dejamos', ['Bad Bunny, Jhay Cortez']),
 ('dolce', ['Bad Bunny, Jhay Cortez']),
 ('quito', ['Bad Bunny, Jhay Cortez']),
 ('mari', ['Bad Bunny, Jhay Cortez', 'BICHOTA']),
 ('bien', ['Bad Bunny, Jhay Cortez', 'Bandido', 'Hawái', 'Ropa Cara']),
 ('mueve', ['Bad Bunny, Jhay Cortez', 'Reloj']),
 ('fenomenal', ['Bad Bunny, Jhay Cortez']),
 ('pa',
  ['Bad Bunny, Jhay Cortez',
   

#### No solo existe una solución

In [ ]:
rdd_letras.flatMap(lambda x: [(palabra, x[0]) for palabra in x[2]]) \
            .groupByKey() \
            .mapValues(lambda x: set(x)).collect()